# YOLO Object Detection: Ground Truth Preparation

## Overview

This notebook prepares the ground-truth dataset used for YOLO object detection.

Main tasks:

1. Load tree crown polygons and tree inventory data.
2. Clean and standardize species labels.
3. Filter target allergenic tree genera (Betula, Alnus, Fraxinus, Quercus, and Platanus).
4. Clean and validate crown polygons.
5. Assign species labels to crown polygons using spatial relationships.
6. Export the processed allergenic crown dataset for YOLO object detection experiments.

## Data Inputs

- Municipal tree inventory (point dataset)
- Tree crown polygons

## Outputs

- enschede_yolo_detection_ground_truth_v01.shp

**Import libraries**

In [ ]:
import os
import geopandas as gpd
from shapely.validation import make_valid
from pathlib import Path
import sys

# Locate project root
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Enable imports from src
sys.path.append(str(PROJECT_ROOT))

from src.paths import RAW_DIR, EXTERNAL_DIR, INTERIM_DIR, PROCESSED_DIR

**Define dataset directory**

In [ ]:
# Load shapefiles
TREE_CROWN_POLYGONS = EXTERNAL_DIR/ "crown_polygons_gt" / "tree_crown.shp" # tree crowns polygons
MUNICIPAL_TREE_INVENTORY = EXTERNAL_DIR/ "municipal_gt" / "Bomen_gbi.shp" # municipal tree points

**Cleaning tree points shapefile**

In [ ]:
# Read a tree points shapefile
points_gdf= gpd.read_file(MUNICIPAL_TREE_INVENTORY)
points_gdf.head()

In [ ]:
# Leave only tree species columns
points_gdf= points_gdf[["geometry", "LATBOOMSOO"]]
points_gdf.head()

In [ ]:
# Create a robust regex pattern to find variations of these species
regex_pattern = r"\b(QUERCUS|BETULA|ALNUS|FRAXINUS|PLATANUS)\b"

# Keep rows that contains target species 
points_gdf= points_gdf[
    points_gdf["LATBOOMSOO"].str.contains(regex_pattern, case= False, na=False, regex=True)
]

In [ ]:
# Clean species name 
points_gdf["species_name"]= (
    points_gdf["LATBOOMSOO"]
    .str.split(" ")
    .str[0]  # Leave only the main genus name 
    .str.title()
)
# Drop original species name column
points_gdf= points_gdf.drop("LATBOOMSOO", axis= "columns")
points_gdf.head()

**Cleaning tree crowns shapefile**

In [ ]:
# Read a tree polygon shapefile
crowns_gdf = gpd.read_file(TREE_CROWN_POLYGONS)
crowns_gdf.head()

In [ ]:
crowns_gdf= crowns_gdf[["geometry", "id"]]
crowns_gdf["id"]= crowns_gdf["id"].astype("int")

In [ ]:
# Make sure that both shapefiles have the same CRS
if crowns_gdf.crs != points_gdf.crs:
    crowns_gdf= crowns_gdf.to_crs(points_gdf.crs)

In [ ]:
# Filter only valid geometries
crowns_gdf.geometry= crowns_gdf.geometry.apply(make_valid)
crowns_gdf= crowns_gdf[crowns_gdf.geometry.is_empty== False]
print(f"The total number of crowns: {len(crowns_gdf)}")

In [ ]:
# Remove duplicate crowns
crowns_gdf["geom_wkb"]= crowns_gdf.geometry.to_wkb()
crowns_gdf= crowns_gdf.drop_duplicates(subset= "geom_wkb").reset_index(drop=True)
crowns_gdf= crowns_gdf.drop(columns=["geom_wkb"])

print(f"The total number of crowns after removing duplicates: {len(crowns_gdf)}")

In [ ]:
# Find crowns that contain a tree point
crowns_with_points = gpd.sjoin(
    crowns_gdf, 
    points_gdf[["species_name", "geometry"]], 
    how="inner", 
    predicate="intersects"
)

crowns_with_points.head()

In [ ]:
# Add points geometry to joined DataFrame
crowns_with_points["point_geom"] = crowns_with_points["index_right"].apply(
    lambda idx: points_gdf.loc[idx, "geometry"]
)

In [ ]:
# Assign species to each crown based on the points inside it
# Group crowns based on their ID
crowns_grouped = crowns_with_points.groupby("id")

rows = []

for crown_id, group in crowns_grouped:
    # Reset index to prevent duplicated index 
    group= group.reset_index(drop=True)
    
    crown_geom = group["geometry"].iloc[0]  

    # CASE 1 – exactly one point in crown
    if len(group) == 1:
        species = group["species_name"].iloc[0]

    # CASE 2 – two points in crown: choose point closest to centroid
    elif len(group) == 2:
        centroid = crown_geom.centroid
        distances = gpd.GeoSeries(group["point_geom"]).distance(centroid)
        species = group.loc[distances.idxmin(), "species_name"]

    # CASE 3 – three or more points: majority vote
    else:
        species = group["species_name"].mode().iloc[0]

    rows.append({
        "id": int(crown_id),
        "species_name": str(species),
        "geometry": crown_geom
    })

# Convert back to GeoDataFrame
crowns_species_gdf = gpd.GeoDataFrame(
    rows,
    geometry="geometry",
    crs=crowns_with_points.crs
)

print(f"Total number of crowns with assigned allergenic species: {len(crowns_species_gdf)}")

In [ ]:
# Save GeoDataFrame as shapefile
YOLO_DETECTION_GROUND_TRUTH = (
    INTERIM_DIR /
    "yolo_detection_gt_prep" /
    "enschede_yolo_detection_ground_truth_v01.shp"
)

crowns_species_gdf.to_file(YOLO_DETECTION_GROUND_TRUTH)